Seal Face Detection and Cropping

In [ ]:
import cv2
import os
from pathlib import Path

# Directories
image_dir = "/content/drive/MyDrive/Capstone/Deakin/furseal/datasets/train/images"
label_dir = "/content/drive/MyDrive/Capstone/Deakin/furseal/datasets/train/labels"
output_crop_dir = "/content/drive/MyDrive/Capstone/Deakin/furseal/cropped_faces"
os.makedirs(output_crop_dir, exist_ok=True)

# Iterate through label files
for label_file in os.listdir(label_dir):
    if not label_file.endswith('.txt'):
        continue

    img_name = label_file.replace(".txt", ".jpg")
    img_path = os.path.join(image_dir, img_name)
    label_path = os.path.join(label_dir, label_file)

    # Read image
    image = cv2.imread(img_path)
    if image is None:
        continue
    height, width = image.shape[:2]

    # Read bounding boxes
    with open(label_path, 'r') as f:
        for idx, line in enumerate(f.readlines()):
            class_id, x_center, y_center, w, h = map(float, line.strip().split())
            # Convert YOLO format to pixel coordinates
            x1 = int((x_center - w / 2) * width)
            y1 = int((y_center - h / 2) * height)
            x2 = int((x_center + w / 2) * width)
            y2 = int((y_center + h / 2) * height)

            # Crop face
            cropped_face = image[y1:y2, x1:x2]
            out_path = os.path.join(output_crop_dir, f"{img_name[:-4]}_{idx}.jpg")
            cv2.imwrite(out_path, cropped_face)

print("✅ Cropping complete.")


✅ Cropping complete.


Feature Extraction (Embeddings with ResNet50)

In [ ]:
import torch
from torchvision import models, transforms
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet50(pretrained=True)
model = torch.nn.Sequential(*list(model.children())[:-1])  # remove last layer
model.to(device).eval()

# Preprocessing
preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Directory
cropped_dir = output_crop_dir
embeddings = []
image_paths = []

# Loop through cropped faces
for fname in os.listdir(cropped_dir):
    if not fname.endswith(".jpg"):
        continue
    path = os.path.join(cropped_dir, fname)
    img = Image.open(path).convert("RGB")
    input_tensor = preprocess(img).unsqueeze(0).to(device)

    with torch.no_grad():
        emb = model(input_tensor).squeeze().cpu().numpy()
        embeddings.append(emb)
        image_paths.append(path)

embeddings = np.array(embeddings)
np.save("/content/drive/MyDrive/Capstone/Deakin/furseal/embeddings/resnet50_embeddings.npy", embeddings)
print("✅ Embeddings saved.")


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 122MB/s]


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Capstone/Deakin/furseal/embeddings/resnet50_embeddings.npy'

Clustering for Pseudo Labels + Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

# Load embeddings
embeddings = np.load("/content/drive/MyDrive/Capstone/Deakin/furseal/embeddings/resnet50_embeddings.npy")

# Reduce dimensionality
pca = PCA(n_components=50, random_state=42)
reduced_embeddings = pca.fit_transform(embeddings)

# Cluster into groups
n_clusters = 10  # Tune this
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
cluster_labels = kmeans.fit_predict(reduced_embeddings)

# Visualize
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_results = tsne.fit_transform(reduced_embeddings)

plt.figure(figsize=(10, 8))
plt.scatter(tsne_results[:, 0], tsne_results[:, 1], c=cluster_labels, cmap='tab10')
plt.title("t-SNE Clustering of Fur Seal Faces")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.colorbar(label="Cluster ID")
plt.grid(True)
plt.show()

# Save pseudo labels
np.save("/content/drive/MyDrive/Capstone/Deakin/furseal/pseudo_labels.npy", cluster_labels)


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Capstone/Deakin/furseal/embeddings/resnet50_embeddings.npy'